# Phase 1: Exploratory Data Analysis

This notebook reproduces the EDA figures and descriptive statistics.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC = os.path.join(BASE, 'data', 'processed')

df = pd.read_parquet(os.path.join(PROC, 'orientation_neuron_summary.parquet'))
bd = np.load(os.path.join(PROC, 'orientation_binned_tuning.npz'))
tuning_mean = bd['tuning_mean']
tuning_sem = bd['tuning_sem']
centers_deg = bd['angle_bin_centers_deg']
dt = np.load(os.path.join(PROC, 'orientation_decoder_targets.npz'))
theta_deg = dt['theta_deg']

print(f'Neurons: {len(df)}, Trials: {len(theta_deg)}')


## Orientation Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(theta_deg, bins=36, color='steelblue', edgecolor='white')
ax.set_xlabel('Stimulus orientation (deg)')
ax.set_ylabel('Trials')
ax.set_title('Trial orientation distribution')
plt.tight_layout()
plt.show()


## Response Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df['mean_response'], bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Mean response'); axes[0].set_title('Mean response')
axes[1].hist(df['response_std'], bins=80, color='darkorange', edgecolor='none')
axes[1].set_xlabel('Response std'); axes[1].set_title('Response std')
plt.tight_layout(); plt.show()


## Reliability Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['split_half_reliability'], bins=60, color='darkorange', edgecolor='white')
ax.set_xlabel('Split-half reliability (r)'); ax.set_ylabel('Neurons')
ax.set_title('Reliability distribution')
plt.tight_layout(); plt.show()


## Preferred Orientation Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['pref_orientation_deg_empirical'], bins=36, color='seagreen')
axes[0].set_xlabel('Pref orientation (deg)'); axes[0].set_title('Empirical')
df_fit = df[df['fit_success']]
axes[1].hist(df_fit['fit_pref_orientation_deg'], bins=36, color='mediumpurple')
axes[1].set_xlabel('Pref orientation (deg)'); axes[1].set_title('Fitted')
plt.tight_layout(); plt.show()


## Tuning Sharpness and Fit Quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_fit['fit_kappa_or_width'], bins=60, color='coral')
axes[0].set_xlabel('Kappa'); axes[0].set_title('Tuning sharpness')
axes[1].hist(df_fit['fit_r2'], bins=60, color='teal')
axes[1].set_xlabel('R-squared'); axes[1].set_title('Fit R-squared')
plt.tight_layout(); plt.show()


## Sharpness vs Reliability

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.hexbin(df_fit['fit_kappa_or_width'].clip(0, 15),
          df_fit['split_half_reliability'], gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_xlabel('Kappa'); ax.set_ylabel('Reliability')
ax.set_title('Sharpness vs reliability')
plt.tight_layout(); plt.show()


## Example Tuning Curves

In [ ]:
def von_mises_tuning(theta, b, a, kappa, theta0):
    return b + a * np.exp(kappa * np.cos(2.0 * (theta - theta0)))

rng = np.random.default_rng(42)
good = df_fit[df_fit['split_half_reliability'] > 0.5].index.values
chosen = rng.choice(good, size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
theta_fine = np.linspace(0, np.pi, 200)
for ax, nid in zip(axes.ravel(), chosen):
    row = df.iloc[nid]
    ax.errorbar(centers_deg, tuning_mean[nid], yerr=tuning_sem[nid],
                fmt='o', ms=3, capsize=2, color='steelblue')
    y_fit = von_mises_tuning(theta_fine, row['fit_baseline'], row['fit_amplitude'],
                             row['fit_kappa_or_width'], np.radians(row['fit_pref_orientation_deg']))
    ax.plot(np.degrees(theta_fine), y_fit, '-', color='tomato', lw=1.5)
    ax.set_title(f'N{nid} R2={row["fit_r2"]:.2f} k={row["fit_kappa_or_width"]:.1f}', fontsize=8)
plt.tight_layout(); plt.show()


## Descriptive Stats

In [ ]:
stats = pd.read_csv(os.path.join(BASE, 'reports', 'tables', 'eda_descriptive_stats.csv'))
stats.T
